# Fire & Smoke Detection — YOLOv8 Training
Trains a YOLOv8 model to detect **fire** and **smoke** in shop floors, stores, and stocking yards.

**Before running:** Go to `Runtime → Change runtime type → T4 GPU`

In [ ]:
# 1. Verify GPU
import subprocess
result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(result.stdout if result.returncode == 0 else 'No GPU — change runtime type to T4 GPU first!')

In [ ]:
# 2. Install dependencies
!pip install ultralytics roboflow --quiet

In [ ]:
# 3. Mount Google Drive (saves model permanently)
from google.colab import drive
drive.mount('/content/drive')

import os
SAVE_DIR = '/content/drive/MyDrive/cctv-safety-training'
os.makedirs(SAVE_DIR, exist_ok=True)
print('Model will be saved to:', SAVE_DIR)

In [ ]:
# 4. Download fire & smoke dataset from Roboflow
from roboflow import Roboflow

API_KEY = 'AlePU7xXgxgnsQkcin0b'  # your Roboflow private API key
rf = Roboflow(api_key=API_KEY)

# --- Choose ONE dataset below ---

# Option A (default): Fire and Smoke Detection — METU (6,391 images | classes: fire, smoke)
project = rf.workspace('middle-east-tech-university').project('fire-and-smoke-detection-hiwia')

# Option B (alternative): Spyrobot — larger dataset (9,749 images | classes: fire, smoke, human)
# Uncomment the line below AND comment out Option A above to switch:
# project = rf.workspace('spyrobot').project('fire-smoke-and-human-detector')

# --------------------------------

# List available versions so you can set the right number in Cell 4b
print('Available versions:')
for v in project.versions():
    print(' -', v)

In [ ]:
# 4b. Pick the latest version number from the output above and set it here
VERSION = 2  # <-- change this if a newer version is listed above

version  = project.version(VERSION)
dataset  = version.download('yolov8', location='/content/fire-smoke-dataset')

print('Dataset path:', dataset.location)

In [ ]:
# 5. Verify dataset — check classes
import yaml
with open(dataset.location + '/data.yaml') as f:
    data_cfg = yaml.safe_load(f)

print('Classes    :', data_cfg['names'])
print('Num classes:', data_cfg['nc'])

# Count images
import glob
train_imgs = glob.glob(dataset.location + '/train/images/*')
val_imgs   = glob.glob(dataset.location + '/valid/images/*')
print(f'Train images: {len(train_imgs)}  |  Val images: {len(val_imgs)}')

In [ ]:
# 6. Train YOLOv8n for fire & smoke detection
#
# yolov8n = nano (fastest, good for real-time CCTV)
# Change to yolov8s for better accuracy if detection is missing subtle smoke

from ultralytics import YOLO

model = YOLO('yolov8n.pt')

results = model.train(
    data=dataset.location + '/data.yaml',
    epochs=60,          # more epochs than PPE since fire/smoke is visually complex
    imgsz=640,
    batch=16,
    device=0,
    project=SAVE_DIR,
    name='fire_smoke_run',
    patience=15,        # early stop if no improvement for 15 epochs
    save=True,
    plots=True,
    # Augmentations useful for fire/smoke (handles varying lighting in stores)
    hsv_h=0.015,
    hsv_s=0.7,
    hsv_v=0.4,
    flipud=0.0,
    fliplr=0.5,
)

print('Training complete!')

In [ ]:
# 7. Evaluate on validation set
best_weights = f'{SAVE_DIR}/fire_smoke_run/weights/best.pt'
model_best   = YOLO(best_weights)
metrics      = model_best.val(data=dataset.location + '/data.yaml', device=0)

print(f'mAP50     : {metrics.box.map50:.3f}')
print(f'mAP50-95  : {metrics.box.map:.3f}')

# Per-class breakdown
for i, name in enumerate(data_cfg['names']):
    print(f'  {name}: AP50 = {metrics.box.ap50[i]:.3f}')

In [ ]:
# 8. Download best.pt to your local machine
from google.colab import files
files.download(best_weights)
print('Downloaded best.pt')
print('Rename it to fire_smoke.pt and place at: models/fire_smoke.pt in your project')

In [ ]:
# 9. Quick visual test on a sample validation image
val_images = glob.glob(dataset.location + '/valid/images/*.jpg')[:1]
if val_images:
    from IPython.display import Image, display
    result = model_best.predict(val_images[0], conf=0.35, save=True, imgsz=640)
    saved  = list(result[0].save_dir.iterdir())[0]
    display(Image(str(saved)))
else:
    print('No validation images found for preview')